🐢 Use the general parameters for an abitrary finetune

🐕 Adjust the parameters and review in weights & biases.

🦁 Adjust the parameters and show change in inference results.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
%pip install trl==0.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 7.4 MB/s eta 0:00:00


In [ ]:
%pip show trl

Name: trl
Version: 0.12.0
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/huggingface/trl
Author: Leandro von Werra
Author-email: leandro.vonwerra@gmail.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: accelerate, datasets, rich, transformers
Required-by: 


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch

device = ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path="HuggingfaceTB/SmolLM2-135M"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="HuggingfaceTB/SmolLM2-135M")
# since we disable chat template for code fullfilling task
# we need to set pad_token due to STTTrain need
tokenizer.pad_token = tokenizer.eos_token
# setup for chat mode in default ChatML template
# notice: use tokenizer.apply_chat_template() for instruct model && tokenizer.chat_template() for self-defined chat
# instruct model: product of base model being instruct fine-tuned
# model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

# comment out the last line and add the below line in code filling task
tokenizer.chat_template = None
# this will ban conversation template

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

In [ ]:
# Let's define LoRA parameters for finetuning
from peft import LoraConfig

# rank for LoRA update matrix (smaller->more compression)
rank_dimension = 6
# scaling factor for LoRA layers (higher->stronger adaption)
lora_alpha = 8
# lora_dropout: dropout probability for LoRA layers
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

# target_modules can be specified:
"""
target_modules=["q_proj", "v_proj"]
target_modules=[
    "q_proj", "k_proj", "v_proj",
    "o_proj",
    "gate_proj", "up_proj", "down_proj"
target_modules=[".*proj.*"]
target_modules=[".*self_attn.*"]
"""
# you can check out what's in the modul by
"""for name, module in model.named_modules:"""

'for name, module in model.named_modules:'

In [ ]:
# test our base model
prompt = "def quicksort(arr):"

# message = [{"role": "user", "content": prompt}]
# formatted_prompt = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt').to(device)
outputs = model.generate(**inputs,
  max_new_tokens=300,
  temperature=0.8,
  top_p=0.9,
  repetition_penalty=1.2,
  do_sample=True,
  eos_token_id=tokenizer.eos_token_id,
)
print(f"Base model outputs: (prompt: {prompt})")
# or
# print("Base model outputs: (prompt: {})".format(prompt))
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Base model outputs: (prompt: def quicksort(arr):)
def quicksort(arr):
    sorted = []

    for i in range (len(arr)):
        x_index, y_index  # index of first and last element to be copied

        if arr[i] > arr[y]:
            swap1=x+1<n-0
                while swaps < n:
                    swap2=(swap1 + j)>=j
                        if swap3>m-(int)(swans/k)+sorted.count((list("".join("%s"*(num * k))))[(int(*maxsort))]):
                            m -= maxSort([min(''.join('%d' % int (*nums).get())],**abs()* num),
                                min([''].append(('%.4f', '-').format(**tmp)), list('_'))+(temp(-todo['**.'+str(l)][:])[-:]) )

                        temp+= nums[:]


#print sortArray();


In [ ]:
# Data Preparation
ds = load_dataset("bigcode/the-stack-smol", data_dir="data/python", split='train')
ds

README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

data/python/data.json:   0%|          | 0.00/87.0M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['content', 'avg_line_length', 'max_line_length', 'alphanum_fraction', 'licenses', 'repository_name', 'path', 'size', 'lang'],
    num_rows: 10000
})

In [ ]:
ds = ds.train_test_split(test_size=0.1)

In [ ]:
ds['train'][0]['content']

"# -*- coding: utf-8 -*-\n# Generated by Django 1.11.9 on 2018-12-06 16:48\nfrom __future__ import unicode_literals\n\nfrom django.db import migrations, models\n\n\nclass Migration(migrations.Migration):\n\n    dependencies = [\n        ('roster', '0021_auto_20180825_1843'),\n    ]\n\n    operations = [\n        migrations.AlterField(\n            model_name='student',\n            name='track',\n            field=models.CharField(choices=[('A', 'Weekly'), ('B', 'Biweekly'), ('C', 'Correspondence'), ('E', 'External'), ('N', 'Not applicable')], max_length=5),\n        ),\n    ]\n"

In [ ]:
def formatting_func(raw_data):
  # model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)
  return raw_data["content"]

In [ ]:
# Configure SFTTrainer
sft_config = SFTConfig(
    output_dir='./SFT_output',
    # lr is constant if you use max_step
    max_steps=300,
    # num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2, # accumulate gradients
    # real batch_size = per_device_train_batch_device * gradient_accumulation_steps
    # trade compute time for memory savings
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    max_grad_norm=0.3,
    learning_rate=3e-4,
    warmup_ratio=0.03, # learning_rate grows linearly and gradually to set value
    lr_scheduler_type="linear",
    logging_steps=50, # this will print logs every 10 steps(loss, lr...)
    save_strategy="epoch",
    bf16=True,
    # eval_strategy='steps',
    # eval_steps=100,
    hub_model_id="SFT_on_SmolTalk",
  )

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    peft_config=peft_config,
    tokenizer=tokenizer,
    eval_dataset=ds["test"],
    formatting_func=formatting_func
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()
trainer.save_model(f"./SFT_on_SmolTalk")

Step,Training Loss
50,1.473400
100,1.410600
150,1.506500
200,1.387100
250,1.457400
300,1.481700


In [ ]:
# test our fine-tuned model
prompt = "def quicksort(arr):"

# message = [{"role": "user", "content": prompt}]
# formatted_prompt = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt').to(device)
outputs_post = model.generate(**inputs,
  max_new_tokens=500,
  temperature=0.8,
  top_p=0.9,
  repetition_penalty=1.2,
  do_sample=True,
  eos_token_id=tokenizer.eos_token_id,
)
print(f"Fine-tuned model outputs: (prompt: {prompt})")
# or
# print("Base model outputs: (prompt: {})".format(prompt))
print(tokenizer.decode(outputs_post[0], skip_special_tokens=True))
print("Base model outputs: (prompt: {})".format(prompt))

In [ ]:
# Merge LoRA adapter into original model
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=args.output_dir,
    torch.dtype=torrch.float16,
    low_cpu_mem_usage=True,
)

merged_model = model.merge_and_unload()
merged_model.save_pretrained(
    args.output_dir, safe_serialization=True, max_shard_size="2GB"
)